# 📈 Estrategia Momentum 120 días — Qlib (paper-trading)

> **Proyecto:** Qlib Work — inversión cuantitativa sistemática
> **Universo:** `sp500_liquid` (292 tickers del S&P 500 con historial desde 2010)
> **Señal:** momentum 120d (retorno acumulado de 120 días) · topk 30
> **Validación:** IC out-of-sample **+0.066** | Backtest **+21.7% anual**, Sharpe **1.07**, DD **−18.6%**
> **Estado:** en paper-trading con €20,000 ficticios (rebalanceo semanal)

Este notebook reproduce la estrategia de extremo a extremo: datos → señal → walk-forward (IC) → backtest → simulación.

> ⚠️ **IMPORTANTE:** este es material de aprendizaje e investigación con datos ficticios. No es una recomendación de inversión.


In [ ]:
# %% Configuración y entorno
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import sys, numpy as np, pandas as pd
import qlib
from qlib.data import D

#QLIB_URI = os.environ.get("QLIB_US_DATA", "/opt/data/profiles/investments/home/.qlib/qlib_data/us_data")
QLIB_URI = os.environ.get("QLIB_US_DATA", "~/.qlib/qlib_data/us_data")

QLIB_URI = "/Users/toni/.qlib/qlib_data/us_data"
UNIVERSE = "sp500_liquid"
MOM_W = 120       # ventana de momentum (días)
TOPK = 30         # número de acciones a seleccionar
START = "2018-01-01"
END = "2026-08-13"

qlib.init(provider_uri=QLIB_URI, region='us')
print("✅ Qlib inicializado")
print(f"   Datos: {QLIB_URI}")

In [ ]:
# %% Cargar el universo y los precios
from qlib.data import D as _D
import os as _os

# Asegurar que el universo sp500_liquid existe (si no, generarlo desde sp500.txt)
_inst_dir = _os.path.join(QLIB_URI, "instruments")
_sp500_liquid = _os.path.join(_inst_dir, "sp500_liquid.txt")
if not _os.path.exists(_sp500_liquid):
    _sp500 = _os.path.join(_inst_dir, "sp500.txt")
    if _os.path.exists(_sp500):
        with open(_sp500) as _f, open(_sp500_liquid, "w") as _out:
            for _line in _f:
                _parts = _line.strip().split("	")
                if len(_parts) >= 3 and _parts[1] <= "2010-01-01" and _parts[2] >= "2026-01-01":
                    _out.write(_line)
        print("⚠️ sp500_liquid.txt no existía → generado desde sp500.txt")
    else:
        print("⚠️ No se encontró sp500.txt ni sp500_liquid.txt. Revisa los datos.")

tickers = _D.list_instruments(_D.instruments(UNIVERSE), as_list=True)
print(f"Tickers en {UNIVERSE}: {len(tickers)}")

# El análisis histórico (walk-forward y backtest) usa SIEMPRE Qlib (historia desde 2018).
# El CSV fresco de prices_live.csv solo se usa para la ÚLTIMA señal / simulación.
close = D.features(tickers, ["$close", "$factor"], start_time=START, end_time="2100-12-31", freq="day")
c = close["$close"].unstack(level=0).sort_index()
f = close["$factor"].unstack(level=0).sort_index()
close = c / f  # precio real = close / factor
close = close.sort_index()
print(f"Usando datos de Qlib (historia completa, última fecha: {close.index[-1].date()})")
close = close.dropna(axis=1, how='all').sort_index()
print(f"Shape: {close.shape} | rango: {close.index[0].date()} → {close.index[-1].date()}")

In [ ]:
# %% Calcular la señal de momentum 120 días
# Momentum = retorno acumulado en la ventana: (precio_hoy / precio_hace_120d) - 1
mom = close / close.shift(MOM_W) - 1

# Señal más reciente (ranking)
latest_mom = mom.iloc[-1].sort_values(ascending=False)
print("Top 10 por momentum 120d (última fecha):")
print(latest_mom.head(10).round(4).to_string())

# Gráfico: evolución del momentum (media del universo)
import matplotlib.pyplot as plt
mom.mean(axis=1).plot(figsize=(12,4), title=f"Momentum {MOM_W}d — media del universo {UNIVERSE}")
plt.ylabel("Momentum"); plt.grid(alpha=0.3); plt.show()

In [ ]:
# %% Validación WALK-FORWARD — IC (correlación señal → retorno futuro)
# Divide en ventanas por año y mide si la señal predice el retorno futuro
fwd = close.shift(-MOM_W) / close - 1   # retorno futuro a 120d

mom_s = mom.stack(dropna=False).rename("mom")
fwd_s = fwd.stack(dropna=False).rename("fwd")
df = pd.concat([mom_s, fwd_s], axis=1).reset_index()
df.columns = ["datetime", "instrument", "mom", "fwd"]
df["datetime"] = pd.to_datetime(df["datetime"])
df["year"] = df["datetime"].dt.year
df = df.dropna()

ic_by_year = df.groupby("year").apply(lambda g: g["mom"].corr(g["fwd"]))
ic_by_year.plot(kind="bar", figsize=(12,4), title="IC del momentum 120d por año (walk-forward OOS)")
plt.axhline(0, color="red", linestyle="--"); plt.ylabel("IC"); plt.grid(alpha=0.3); plt.show()

ic_mean = ic_by_year.mean()
print(f"IC medio out-of-sample: {ic_mean:.4f}")
print("Veredicto:", "✅ ALPHA REAL (IC > 0.02)" if ic_mean > 0.02 else "⚠️ Sin alpha robusto")

In [ ]:
# %% Backtest con Qlib (TopkDropout + costes Interactive Brokers)
from qlib.backtest import backtest

IB = dict(open_cost=0.0004, close_cost=0.0006, min_cost=1.0,
          limit_threshold=0.095, deal_price="close")

# Señal en formato Qlib (MultiIndex datetime, instrument)
sig = mom.stack().dropna()
sig.name = "momentum"
sig = sig.reset_index()
sig.columns = ["datetime", "instrument", "momentum"]
sig["datetime"] = pd.to_datetime(sig["datetime"])
sig = sig.set_index(["datetime", "instrument"])["momentum"].sort_index()

strategy = {
    "class": "TopkDropoutStrategy",
    "module_path": "qlib.contrib.strategy",
    "kwargs": {"signal": sig, "topk": TOPK, "n_drop": 5, "only_tradable": True, "hold_thresh": 20},
}
executor = {
    "class": "SimulatorExecutor",
    "module_path": "qlib.backtest.executor",
    "kwargs": {"time_per_step": "week", "generate_portfolio_metrics": True},
}

report_normal, indicator = backtest(
    start_time="2018-01-01", end_time="2026-08-07",
    strategy=strategy, executor=executor,
    benchmark="^NDX", account=100000, exchange_kwargs=IB,
)

freq_key = list(report_normal.keys())[0]
dfr = report_normal[freq_key][0]
pv = dfr["account"].values
ret = pd.Series(pv).pct_change().dropna().values
mean_ann = float(np.mean(ret)) * 52
vol_ann = float(np.std(ret)) * np.sqrt(52)
cum = np.cumprod(1 + ret)
dd = cum / np.maximum.accumulate(cum) - 1
sharpe = mean_ann / vol_ann if vol_ann else np.nan

print("📊 RESULTADOS BACKTEST (retorno absoluto)")
print(f"  Valor final: ${pv[-1]:,.0f} (inicial ${pv[0]:,.0f})")
print(f"  Retorno anualizado: {mean_ann*100:+.2f}%")
print(f"  Volatilidad anual:  {vol_ann*100:.2f}%")
print(f"  Max drawdown:       {dd.min()*100:.2f}%")
print(f"  Sharpe:             {sharpe:.3f}")

# Curva del portfolio
pd.Series(pv, index=dfr.index).plot(figsize=(12,4), title="Valor del portfolio (backtest)")
plt.ylabel("USD"); plt.grid(alpha=0.3); plt.show()

In [ ]:
# %% Simulación paper-trading (dinero ficticio) — acciones fraccionales y enteras
import os, json
sim_dir = os.path.join(os.path.dirname(os.path.abspath('momentum_120d.ipynb')), 'simulation')
state_file = os.path.join(sim_dir, 'state.json')

if os.path.exists(state_file):
    with open(state_file) as f:
        state = json.load(f)
    pv_sim = float(state["cash_usd"])
    for t, pos in state["positions"].items():
        cur = close[t].iloc[-1] if t in close.columns else float(pos["entry_price"])
        pv_sim += float(pos["shares"]) * cur
    euro_usd = float(state["euro_usd"])
    print("📈 ESTADO PAPER-TRADING (dinero ficticio)")
    print("  Capital inicial:  EUR {:.0f}".format(float(state["start_capital_eur"])))
    print("  Valor actual:     ${:,.0f} = EUR {:,.0f}".format(pv_sim, pv_sim/euro_usd))
    print("  P&L:              ${:+,.0f} ({:+.2f}%)".format(
        pv_sim-float(state["start_capital_usd"]),
        (pv_sim-float(state["start_capital_usd"]))/float(state["start_capital_usd"])*100))
    print("  Fecha:            {} | Posiciones: {}".format(state["date"], len(state["positions"])))

    # Tabla completa: fraccional y entera (como el script simulate.py)
    print("")
    print("="*62)
    print("POSICIONES — fraccionales y redondeadas a entero")
    print("="*62)
    print("{:<6}{:>8}{:>7}{:>9}{:>10}{:>9}".format("Ticker","Fracc.","Entero","Precio","CosteFr$","CosteEn$"))
    print("-"*62)
    tf = 0.0; te = 0.0
    rows = []
    for t, pos in state["positions"].items():
        sh = float(pos["shares"])
        pr = close[t].iloc[-1] if t in close.columns else float(pos["entry_price"])
        ent = round(sh)
        cf = sh * pr
        ce = ent * pr
        tf += cf; te += ce
        rows.append((t, sh, ent, pr, cf, ce))
    rows.sort(key=lambda r: -r[3])  # ordenar por precio desc
    for t, sh, ent, pr, cf, ce in rows:
        print("{:<6}{:>8.2f}{:>7}{:>9.2f}${:>8,.0f}${:>8,.0f}".format(t, sh, ent, pr, cf, ce))
    print("-"*62)
    print("{:<6}{:>8}{:>7}{:>9}${:>8,.0f}${:>8,.0f}".format("TOTAL","","","",tf,te))
    print("")
    print("  Coste fraccionario: ${:,.0f} = EUR {:,.0f}".format(tf, tf/euro_usd))
    print("  Coste redondeado:   ${:,.0f} = EUR {:,.0f}".format(te, te/euro_usd))

    # Comparativa de costes IB (fraccional vs entera) — lógica real de IB (tiered)
    # (funciones locales para no re-inicializar Qlib con un import del script)
    IB_RATE_PER_SHARE = 0.0035
    IB_MIN_PER_ORDER = 0.35
    def _ib_buy_cost(shares, price):
        return max(IB_MIN_PER_ORDER, shares * IB_RATE_PER_SHARE)
    def _ib_trades_cost(buys):
        return sum(_ib_buy_cost(sh, pr) for sh, pr in buys)

    opt_frac = []
    opt_ent = []
    for t, pos in state["positions"].items():
        sh = float(pos["shares"])
        pr = close[t].iloc[-1] if t in close.columns else float(pos["entry_price"])
        ent = round(sh) or 1
        opt_frac.append((t, sh, pr))
        opt_ent.append((t, ent, pr))

    def _option_total(opt):
        value = sum(sh*pr for _, sh, pr in opt)
        ib = _ib_trades_cost([(sh, pr) for _, sh, pr in opt])
        return value, ib, value+ib

    val_f, ib_f, tot_f = _option_total(opt_frac)
    val_e, ib_e, tot_e = _option_total(opt_ent)

    print("")
    print("="*62)
    print("🔀 COMPARATIVA COSTES IB — fraccional vs entera (compra)")
    print("="*62)
    print(f"  {'Opción':22}{'Acciones':>11}{'CosteIB':>9}{'Total$':>15}")
    print("-"*62)
    print(f"  {'Fraccionaria (30)':22}{sum(sh for _,sh,_ in opt_frac):>11,.2f}${ib_f:>8,.2f}${tot_f:>14,.2f}")
    print(f"  {'Entera (30)':22}{sum(sh for _,sh,_ in opt_ent):>11,.0f}${ib_e:>8,.2f}${tot_e:>14,.2f}")
    print("-"*62)
    diff = tot_e - tot_f
    print(f"  💡 Diferencia (entera − fracc): ${diff:+,.0f} "
          f"({'cuesta MÁS' if diff>0 else 'cuesta MENOS'})")
    print("  📌 Con fraccionarias de IB puedes replicar la cartera casi exacta;")
    print("     con enteras (+${:,.0f}) inviertes más capital.".format(abs(diff)))
else:
    print("No hay estado de simulación aún. Ejecuta: python work/estrategias/simulation/simulate.py --reset")


---
## 📋 Notas de uso

**Ejecutar el notebook:**
```bash
cd /opt/data/qlib
/opt/data/qlib-venv/bin/jupyter lab            # JupyterLab
# o
/opt/data/qlib-venv/bin/jupyter notebook        # Notebook clásico
```

**Kernel:** usa el Python del venv (`qlib-venv`) que tiene Qlib instalado.

**Datos frescos (opcional):** ejecuta `python work/estrategias/simulation/update_data_light.py` antes para usar los precios más recientes de Yahoo en lugar de los datos históricos de Qlib.

**Reproducir la simulación:** `python work/estrategias/simulation/simulate.py [--reset]`

---

*Notebook de referencia del proyecto Qlib Work. La estrategia está validada con walk-forward (IC OOS 0.066) y en paper-trading.*
